# Simplified Chinese support and reach on Steam

**DATA2010 Data Science Programming · final project**

The notebook follows the same sections as the README.

| Section | What it does | Output |
| :--- | :--- | :--- |
| I. Setup and how to reproduce | how to get the project and run it | |
| II.1 Business understanding | the question | |
| II.2 Data collection | read the two raw files | |
| II.3 Data processing | 3.1 clean · 3.2 SQ1 groups · 3.3 SQ2 groups · 3.4 save the tables | `pcd_games_clean`, `pcd_sq1_groups`, `pcd_sq2_groups` |
| II.4 Data analysis / EDA | describe the three tables | `report/figures/` |
| II.5 Data modelling | compare the groups, answer SQ1-SQ3 | `report/figures/` |
| II.6 Model deployment | | |

**How to read it**
- Each code cell = one step, labelled 2.1, 2.2, ..., 3.1.1, 3.1.2, ...; in section 3.1, each problem of a step has a letter (3.1.1a, 3.1.1b, ...) used in the table and in the code comments.
- Its first lines say: `Reason` (why), `Rule` (what is done), `Not used ... because` (what was left out).
- `DATA ISSUE` = a problem found in the data.

# I. Setup and how to reproduce

**Needed:** [Git](https://git-scm.com) with [Git LFS](https://git-lfs.com) (the Windows installer of Git already includes it), Python 3.12 or 3.13.

**Steps**
1. `git lfs install` (once per computer)
2. `git clone https://github.com/theonlypgk/data2010-steam-chinese.git`
3. `cd data2010-steam-chinese` then `pip install -r requirements.txt`
4. Open `scripts/steam_chinese_analysis.ipynb` and choose *Run All*.

- `data/raw/games_a.csv` is only a few lines of text (not ~400 MB)? → run `git lfs pull` in the project folder.

In [ ]:
# I.a · Import the libraries and name the folders
# Reason: three libraries are enough; paths are relative to this notebook (scripts/)

import re                     # text patterns (clean the list cells, find Chinese characters)

import numpy as np
import pandas as pd

RAW = "../data/raw/"                          # the two input files
PROCESSED_CSV = "../data/processed/csv/"      # output tables
FIGURES = "../report/figures/"                # saved figures
PREFIX = "pcd_"                               # file prefix of every output table
print("pandas", pd.__version__, "| numpy", np.__version__)

# II. Data science life cycle

## 1.​ Business understanding

**Main question:** should a Steam game support Simplified Chinese, for which genres, and when?

| | Question |
|---|---|
| SQ1 | How much larger is the reach of games with Simplified Chinese, and in which genres? |
| SQ2 | Does adding Simplified Chinese after launch still increase reach, and in which genres? |
| SQ3 | For each genre: translate at launch, later, or not a priority? |

Context and the definition of reach: see the [README, II.1](https://github.com/theonlypgk/data2010-steam-chinese#1-business-understanding).

## 2.​ Data collection

- **Data:** Steam Games Dataset by FronkonGames (Hugging Face `FronkonGames/steam-games-dataset`), one row per game.
- **Two snapshots:** A = 2026-06-20, C = 2024-01-11, saved in `data/raw/` → same input on every run.
- **DATA ISSUE:** header of A merges two names (`DiscountDLC count`) → later columns shifted → repaired before reading.

### 2.1 The two snapshots

In [ ]:
# 2.1 · The two snapshots
# Reason: the dataset keeps changing -> the two files used are saved in data/raw
# Source: Hugging Face FronkonGames/steam-games-dataset, file games.csv
#   A = commit 52ba62abfa707a2fa31f4d147c2777a9d9fbd2da; C = commit aefdeb1ff1a739b92ac5338682cc2610a0328505

FILE_A = RAW + "games_a.csv"
FILE_C = RAW + "games_c.csv"
SNAPSHOT_DATE_A = "2026-06-20"                # day snapshot A was taken
SNAPSHOT_DATE_C = "2024-01-11"                # day snapshot C was taken

### 2.2 Read the two snapshots

In [ ]:
# 2.2 · Read the two snapshots
# DATA ISSUE: header of A merges two names ("DiscountDLC count") -> split it back, else columns shift
# Rule: read everything as text (dtype=str); fix types in 3.1.1

raw_c = pd.read_csv(FILE_C, dtype=str)

names_a = pd.read_csv(FILE_A, nrows=0).columns.tolist()     # header only
i = names_a.index("DiscountDLC count")
names_a[i:i + 1] = ["Discount", "DLC count"]                 # 1 name -> 2 names
raw_a = pd.read_csv(FILE_A, names=names_a, skiprows=1, dtype=str)

### 2.3 Keep the 12 columns the analysis uses

In [ ]:
# 2.3 · Keep the 12 columns the analysis uses, with English names
# Reason: 28 of the 40 columns do not serve the questions

# Columns kept -> used for:
#   AppID                 -> id; join C to A
#   Name                  -> check results by eye
#   Release date          -> year band; ">= 1 year old" filter
#   Price, Discount       -> price tier (list price)
#   Supported languages   -> treatment; other languages; Chinese-first rule 1; languages added (3.3)
#   Full audio languages  -> voice-over signal, only for the stricter filter of 5.1b
#   Positive, Negative    -> outcome (reach)
#   Developers            -> Chinese-first rule 2
#   Genres                -> Early Access flag; the list of Steam genres (3.1.6)
#   Tags                  -> genre group
# Not kept because:
#   Estimated owners                         -> ranges too wide
#   scores, playtime, CCU, achievements      -> measure quality/engagement, not reach
#   Publishers                               -> often regional partners, not the studio
#   Categories, age, DLC, platforms, notes   -> not part of the design
#   descriptions, links, media               -> no numbers
RAW_TO_CLEAN = {
    "AppID": "app_id",
    "Name": "name",
    "Release date": "release_date",
    "Price": "price",
    "Supported languages": "supported_languages",
    "Full audio languages": "full_audio_languages",
    "Positive": "positive",
    "Negative": "negative",
    "Developers": "developers",
    "Genres": "genres",
    "Tags": "tags",
}

raw_a = raw_a[list(RAW_TO_CLEAN) + ["Discount"]].rename(columns=RAW_TO_CLEAN | {"Discount": "discount_pct"})
raw_c = raw_c[list(RAW_TO_CLEAN)].rename(columns=RAW_TO_CLEAN)
raw_c["discount_pct"] = "0"                    # C has no Discount column
print(f"snapshot A: {len(raw_a):,} games x {raw_a.shape[1]} columns")
print(f"snapshot C: {len(raw_c):,} games x {raw_c.shape[1]} columns")
raw_a.head(3)

## 3.​ Data processing

| Part | What | Output |
|---|---|---|
| 3.1 Clean data | fix data issues, add analysis columns, filter | `pcd_games_clean` |
| 3.2 SQ1 groups | similar games, with vs without Chinese | `pcd_sq1_groups` |
| 3.3 SQ2 groups | similar games, added Chinese vs did not | `pcd_sq2_groups` |
| 3.4 Save the tables | write the output files | |

- **Rule:** snapshots A and C are cleaned the same way.

### 3.1 Clean data

Three blocks. Each step = a table of problems (a, b, c, ...), then its code cell; the code marks each problem with its label (e.g. `# 3.1.1e`).

| Block | Steps | Output |
|---|---|---|
| Fix and add columns | 3.1.1-3.1.8 | `feat_a`, `feat_c` (all games, new columns) |
| Filter the rows | 3.1.9-3.1.10 | the filters |
| Build the table | 3.1.11-3.1.12 | `games_clean`, the funnel |

#### Fix and add columns (3.1.1-3.1.8)

##### 3.1.1 · Fix types and wrong single values

| # | Problem | Solution | Output |
|---|---|---|---|
| a | every value is read as text | turn ids, review counts and prices into numbers | `"1000"` → `1000` |
| b | some games have no id, or an id that is not a number | drop the row | `app_id = ""` → row dropped |
| c | the same game appears twice (repeated id) | keep the first row | 2 rows with one id → 1 row |
| d | missing review count | count it as 0 | `positive = ""` → `0` |
| e | **DATA ISSUE:** snapshot A stores the sale price | list price = `price / (1 - discount / 100)` | Hades `7.49` at `-70%` → `24.99` |
| f | a 100% discount would divide by zero | cap the discount at 99% | `100` → `99` |
| g | dates come in several formats; some cannot be read | read every format; unreadable → missing | `"Aug 1, 2023"` → `2023-08-01`; `"Coming soon"` → `NaT` |
| h | the analysis needs the release year and the age of the game | 2 new columns from the date | `release_year = 2023`, `days_since_release` = days from release to the snapshot |

In [ ]:
# 3.1.1 · Fix types and wrong single values (problems a-h above)

def fix_types(trim, snapshot_date):
    """Give one snapshot real types, one row per game, list prices and dates.

    Args:
        trim: one snapshot from 2.3 (every value is text).
        snapshot_date: day the snapshot was taken, "YYYY-MM-DD".

    Returns:
        A new DataFrame, one row per app_id:
        app_id, positive, negative -> int64; price -> list price (float);
        release_date -> datetime (NaT if unreadable); new columns
        price_at_snapshot, release_year, days_since_release.
    """
    df = trim.copy()
    # 3.1.1a, b · id: text -> number; not a number -> missing
    df["app_id"] = pd.to_numeric(df["app_id"], errors="coerce")
    # 3.1.1b · no id -> row dropped;  3.1.1c · repeated id -> keep the first row
    df = df.dropna(subset=["app_id"]).drop_duplicates("app_id")
    df["app_id"] = df["app_id"].astype("int64")
    # 3.1.1a, d · review counts: text -> number; missing -> 0
    for col in ("positive", "negative"):
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("int64")
    # 3.1.1a · price: text -> number (the price on the snapshot day)
    df["price_at_snapshot"] = pd.to_numeric(df["price"], errors="coerce")
    # 3.1.1f · discount: missing -> 0; capped at 99% because: 100% -> divide by zero
    discount = df["discount_pct"] if "discount_pct" in df else pd.Series(0, index=df.index)
    df["discount_pct"] = pd.to_numeric(discount, errors="coerce").fillna(0).clip(0, 99)
    # 3.1.1e · sale price -> list price
    df["price"] = (df["price_at_snapshot"] / (1 - df["discount_pct"] / 100)).round(2)
    # 3.1.1g · "Aug 1, 2023" or "Aug 2023" -> date; unreadable -> NaT (missing)
    df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce", format="mixed")
    # 3.1.1h · release year, and days from release to the snapshot
    df["release_year"] = df["release_date"].dt.year.astype("Int64")
    df["days_since_release"] = (pd.Timestamp(snapshot_date) - df["release_date"]).dt.days
    return df

##### 3.1.2 · Clean the list cells

| # | Problem | Solution | Output |
|---|---|---|---|
| a | languages, tags and genres are text that looks like a list | turn the text into a Python list | `"['English', 'French']"` → `["English", "French"]` |
| b | two ways of writing the list | handle both | `"English,French"` → `["English", "French"]` |
| c | **DATA ISSUE:** store-page junk (BBCode, HTML) | remove the junk first | `"English[b]*[/b]"` → `["English"]` |
| d | the same item written twice | drop repeats, keep the order (tags are sorted by votes) | `["English", "English"]` → `["English"]` |
| e | empty cell | empty list | `NaN` → `[]` |

In [ ]:
# 3.1.2 · Turn the list-like text cells into clean lists (problems a-e above)

# 3.1.2c · store-page junk: BBCode tags, HTML tags/entities, "*", line breaks, "#lang_..." codes
_TAG_JUNK = re.compile(r"\[/?[a-zA-Z]+\]|<[^>]+>|&amp;|&lt;|&gt;|\*|\r|\n|#lang_\w+")


def parse_list(value):
    """Turn one list-like text cell into a clean list.

    Args:
        value: a cell such as "['English', 'French']", "A,B", "English[b]*[/b]" or NaN.

    Returns:
        List of unique strings in the original order, e.g. ["English", "French"];
        [] for a missing cell.
    """
    if pd.isna(value):
        return []                                          # 3.1.2e · empty cell -> []
    text = _TAG_JUNK.sub("", str(value)).strip()           # 3.1.2c · remove the junk first
    items = text.strip("[]").split(",")                    # 3.1.2a, b · "['A', 'B']" or "A,B" -> pieces
    out = []
    for item in items:
        item = item.strip().strip("'\"").strip()           # 3.1.2a · drop spaces and quotes
        if item and item != "nan" and item not in out:     # 3.1.2d · drop empties and repeats, keep the order
            out.append(item)
    return out

##### 3.1.3 · Outcome, treatment, other languages

| # | Problem | Solution | Output |
|---|---|---|---|
| a | Steam publishes no sales | reach = number of reviews | `900 + 100` → `total_reviews = 1000` |
| b | need the treatment: Simplified Chinese or not | look for it in the language list | `[..., "Simplified Chinese"]` → `has_zh_simplified = True` |
| c | 3.1.4 and 3.1.5 need Traditional Chinese and English | look for them too | `has_zh_traditional`, `has_english` (True/False) |
| d | big games ship in many languages → they are both translated and reviewed more | count the other languages (not Chinese, not English) | `["English", "French", "German", "Simplified Chinese"]` → `n_other_langs = 2` |
| e | 3.3 needs to know which languages a game **added** between C and A | keep every language in one text column | `all_langs = "English\|French\|German\|Simplified Chinese"` |

In [ ]:
# 3.1.3 · Outcome, treatment and the language columns (problems a-e above)

ZH_SIMPLIFIED = "Simplified Chinese"
ZH_TRADITIONAL = "Traditional Chinese"
ENGLISH = "English"


def add_outcome_and_languages(df, langs):
    """Add the outcome and the language columns.

    Args:
        df: snapshot after fix_types.
        langs: Series of clean language lists (parse_list of supported_languages).

    Returns:
        The same DataFrame with new columns:
        total_reviews (int), has_zh_simplified, has_zh_traditional, has_english (bool),
        n_other_langs (int), all_langs (str, e.g. "English|French").
    """
    df["total_reviews"] = df["positive"] + df["negative"]                        # 3.1.3a · reach
    df["has_zh_simplified"] = langs.map(lambda x: ZH_SIMPLIFIED in x)            # 3.1.3b · treatment
    df["has_zh_traditional"] = langs.map(lambda x: ZH_TRADITIONAL in x)          # 3.1.3c
    df["has_english"] = langs.map(lambda x: ENGLISH in x)                        # 3.1.3c
    chinese = {ZH_SIMPLIFIED, ZH_TRADITIONAL}
    df["n_other_langs"] = langs.map(lambda x: len(set(x) - chinese - {ENGLISH}))   # 3.1.3d · set difference
    df["all_langs"] = langs.map(lambda x: "|".join(sorted(x)))                   # 3.1.3e · "English|French|..."
    return df

##### 3.1.4 · Flag the Chinese-first games (removed in 3.1.10)

| # | Problem | Solution | Output |
|---|---|---|---|
| a | for a Chinese studio, Chinese is the original language, not a translation → outside the question | flag the game if rule 1 **or** rule 2 is true | `is_cn_game = True` |
| b | game with Chinese text but no English text | rule 1 | `["Simplified Chinese"]` → `cn_text = True` |
| c | game with English, made by a Chinese studio | rule 2: every developer name in Chinese characters | developer `"鬼谷工作室"` → `cn_dev_name = True` |
| d | Japanese studio names also use Chinese characters (kanji) | a developer cell with kana (Japanese only) does not count | `"株式会社ゲームフリーク"` → `cn_dev_name = False` |
| e | one developer cell lists several names | split on commas | `"A, B"` → `["A", "B"]` |
| f | a foreign game can list one Chinese co-developer | require **every** developer to match | `["Ubisoft", "腾讯"]` → `False` |
| g | 5.1b tests a stricter filter | also compute 2 signals that are not used to filter | `cn_dev_kana`, `cn_audio` (True/False) |

- **Not used:** hand-made lists of cities or companies (dataset too large to check row by row); voice-over languages (Japanese games are dubbed in Chinese too); publishers (often regional partners).
- **Missed:** Chinese studios with an English name and an English release (Black Myth: Wukong).

In [ ]:
# 3.1.4 · Flag the Chinese-first games (problems a-g above)

_HAN = re.compile(r"[一-鿿]")       # a Chinese character (also used in Japanese kanji)
_KANA = re.compile(r"[぀-ヿ]")      # hiragana or katakana: only Japanese uses them


def split_developers(text):
    """Split a developer cell into developer names.

    Args:
        text: developer cell, e.g. "A, B".

    Returns:
        List of names, e.g. ["A", "B"].
    """
    return [p.strip() for p in str(text).split(",") if p.strip()]   # 3.1.4e · split on commas


def all_match(developers, pattern):
    """Check that every developer name matches a pattern.

    Args:
        developers: list of names from split_developers.
        pattern: compiled regex, e.g. _HAN (Chinese characters).

    Returns:
        True if the list is not empty and every name matches; otherwise False.
    """
    return bool(developers) and all(pattern.search(d) for d in developers)   # 3.1.4f · every name, not any


def add_chinese_game_flags(df, audio):
    """Flag the Chinese-first games.

    Args:
        df: snapshot after add_outcome_and_languages.
        audio: Series of clean voice-over language lists.

    Returns:
        The same DataFrame with new columns:
        cn_text, cn_dev_name (bool, the 2 rules), cn_dev_kana, cn_audio (bool, not used to filter),
        is_cn_game (bool, cn_text or cn_dev_name).
    """
    developers = df["developers"].fillna("").astype(str)
    han_name = developers.map(split_developers).map(lambda d: all_match(d, _HAN))   # 3.1.4c, e, f
    kana_name = developers.str.contains(_KANA)                                      # 3.1.4d · any kana

    df["cn_text"] = (df["has_zh_simplified"] | df["has_zh_traditional"]) & ~df["has_english"]  # 3.1.4b · rule 1
    df["cn_dev_name"] = han_name & ~kana_name                                       # 3.1.4c, d · rule 2
    df["is_cn_game"] = df["cn_text"] | df["cn_dev_name"]                             # 3.1.4a · rule 1 or 2

    # 3.1.4g · not used to filter; only for the stricter filter of 5.1b
    chinese = {ZH_SIMPLIFIED, ZH_TRADITIONAL}
    df["cn_dev_kana"] = han_name & kana_name                                        # names with kana = Japanese
    df["cn_audio"] = audio.map(lambda x: bool(set(x) & chinese) and ENGLISH not in x)   # Chinese voice-over, no English
    return df

##### 3.1.5 · Flag Early Access and Traditional-Chinese-only games

| # | Problem | Solution | Output |
|---|---|---|---|
| a | Early Access = unfinished game | flag it, remove in 3.1.10 | `is_early_access = True` |
| b | Traditional Chinese only = neither "with" nor "without" Simplified Chinese | flag it, remove in 3.1.11 | `is_traditional_only = True` |

In [ ]:
# 3.1.5 · Flag Early Access and Traditional-Chinese-only games (problems a-b above)

def add_filter_flags(df, genres):
    """Flag the games removed later in 3.1.10 and 3.1.11.

    Args:
        df: snapshot after add_outcome_and_languages.
        genres: Series of clean genre lists.

    Returns:
        The same DataFrame with new columns is_early_access and is_traditional_only (bool).
    """
    df["is_early_access"] = genres.map(lambda x: "Early Access" in x)                   # 3.1.5a
    df["is_traditional_only"] = df["has_zh_traditional"] & ~df["has_zh_simplified"]     # 3.1.5b
    return df

##### 3.1.6 · One genre group per game

| # | Problem | Solution | Output |
|---|---|---|---|
| a | which genres? a hand-made tag → genre list cannot be checked against 125k games | genre list = Steam's own genre labels, taken from the `genres` column by code | 9 genres: Action, Adventure, Casual, Massively Multiplayer, Racing, RPG, Simulation, Sports, Strategy |
| b | some Steam genre labels are not a type of gameplay | drop Indie, Early Access, Free to Play | "Indie" → not a genre |
| c | **DATA ISSUE:** one label written two ways: "Free To Play" (genres of A), "Free to Play" (genres of C, tags) | compare labels in lower case | both dropped |
| d | rare labels (content warnings, software tools) cannot form comparison groups | keep labels on >= 2% of the games | "Violent", "Utilities" → not genres |
| e | each game has several genres and dozens of user tags | the first user tag that is a genre wins (Steam orders tags by votes) | `["Indie", "Adventure", "Puzzle", "Action"]` → `"Adventure"` |
| f | no tag is a genre | missing, removed in 3.1.10 | `["Indie", "Pixel Graphics"]` → `None` |

In [ ]:
# 3.1.6 · Give each game one genre group (problems a-f above)

NOT_GAMEPLAY = {"indie", "early access", "free to play"}   # 3.1.6b · labels that are not a type of gameplay
MIN_GENRE_SHARE = 0.02                                       # 3.1.6d · a label on fewer games cannot form groups


def steam_genres(genre_lists, min_share=MIN_GENRE_SHARE):
    """Build the genre list from Steam's own genre labels.

    Args:
        genre_lists: Series of clean genre lists (parse_list of the genres column).
        min_share: smallest share of games a label must appear on.

    Returns:
        Sorted list of genre names, e.g. ["Action", "Adventure", ...]: the labels on
        >= min_share of the games, without the labels in NOT_GAMEPLAY (any upper/lower case).
    """
    share = genre_lists.explode().value_counts() / len(genre_lists)   # 3.1.6a · share of games with each label
    return sorted(
        g for g, s in share.items()
        if s >= min_share                       # 3.1.6d · not rare
        and g.lower() not in NOT_GAMEPLAY       # 3.1.6b, c · not gameplay; lower case -> both spellings
    )


GENRES = steam_genres(raw_a["genres"].map(parse_list))   # 3.1.6a · from snapshot A; used for A and C
print(f"{len(GENRES)} genres:", ", ".join(GENRES))


def first_genre(tags):
    """Give a game one genre group.

    Args:
        tags: clean list of user tags, most voted first.

    Returns:
        The first tag that is in GENRES, e.g. "RPG"; None if no tag is a genre.
    """
    for tag in tags:
        if tag in GENRES:
            return tag                          # 3.1.6e · first tag that is a genre
    return None                                 # 3.1.6f · no tag is a genre

##### 3.1.7 · Bin and encode price, year, other languages

| # | Problem | Solution | Output |
|---|---|---|---|
| a | prices take too many values to form groups | 5 price tiers | `24.99` → `price_tier = 5` |
| b | release year is too detailed | 4 year bands | `2019` → `year_band = 2` |
| c | number of other languages is too detailed | 3 bins | `7` → `n_other_langs_bin = 3` |
| d | text labels sort wrongly ("10to20" before "5to10") | save each bin as its rank 1, 2, 3, ... | `"5to10"` → `3`, `"10to20"` → `4` |

| Code | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| `price_tier` | free | < \$5 | \$5-10 | \$10-20 | > \$20 |
| `year_band` | <=2017 | 2018-2019 | 2020-2021 | 2022+ | |
| `n_other_langs_bin` | 0 | 1-4 | 5+ | | |

- Genre is not encoded because: genres have no order.

In [ ]:
# 3.1.7 · Bin price, release year and other languages, and encode the bins (problems a-d above)

PRICE_BINS = [-0.001, 0.0, 4.999, 9.999, 19.999, float("inf")]    # 3.1.7a · USD, list price
PRICE_LABELS = ["free", "lt5", "5to10", "10to20", "gt20"]           #          codes 1-5
YEAR_BAND_BINS = [0, 2017, 2019, 2021, 2100]                        # 3.1.7b
YEAR_BAND_LABELS = ["<=2017", "2018-2019", "2020-2021", "2022+"]    #          codes 1-4
OTHER_LANG_BINS = [-1, 0, 4, 10_000]                                # 3.1.7c
OTHER_LANG_LABELS = ["0", "1-4", "5+"]                              #          codes 1-3


def to_code(values, bins):
    """Put numbers into bins and return the bin number.

    Args:
        values: Series of numbers, e.g. prices.
        bins: bin edges, e.g. PRICE_BINS.

    Returns:
        Series of Int64 codes 1, 2, 3, ... (missing stays missing).
    """
    # 3.1.7d · labels=False gives the bin position 0, 1, 2, ...; +1 makes it 1, 2, 3, ...
    return (pd.cut(values.astype("float"), bins, labels=False) + 1).astype("Int64")


def add_codes(df):
    """Add the binned confounders as codes.

    Args:
        df: snapshot with price, release_year and n_other_langs.

    Returns:
        The same DataFrame with new columns price_tier (1-5), year_band (1-4),
        n_other_langs_bin (1-3), all Int64.
    """
    df["price_tier"] = to_code(df["price"], PRICE_BINS)                       # 3.1.7a
    df["year_band"] = to_code(df["release_year"], YEAR_BAND_BINS)             # 3.1.7b
    df["n_other_langs_bin"] = to_code(df["n_other_langs"], OTHER_LANG_BINS)   # 3.1.7c
    return df

##### 3.1.8 · Run on both snapshots

| # | Problem | Solution | Output |
|---|---|---|---|
| a | A and C must be cleaned the same way | one function `add_features` = steps 3.1.1-3.1.7, run on both | `feat_a` 125,855 games, `feat_c` 85,103 games (no game removed yet) |

In [ ]:
# 3.1.8 · Run: clean both snapshots the same way (problem a above)

def add_features(trim, snapshot_date):
    """Run steps 3.1.1-3.1.7 on one snapshot.

    Args:
        trim: one snapshot from 2.3.
        snapshot_date: day the snapshot was taken, "YYYY-MM-DD".

    Returns:
        DataFrame, one row per game (no game removed yet), with every column
        added in 3.1.1-3.1.7.
    """
    df = fix_types(trim, snapshot_date)                               # 3.1.1
    langs = df["supported_languages"].map(parse_list)                 # 3.1.2
    audio = df["full_audio_languages"].map(parse_list)
    tags = df["tags"].map(parse_list)
    genres = df["genres"].map(parse_list)
    df = add_outcome_and_languages(df, langs)                         # 3.1.3
    df = add_chinese_game_flags(df, audio)                            # 3.1.4
    df = add_filter_flags(df, genres)                                 # 3.1.5
    df["genre_group"] = tags.map(first_genre)                         # 3.1.6
    return add_codes(df)                                              # 3.1.7


# 3.1.8a · the same function for both snapshots
feat_a = add_features(raw_a, SNAPSHOT_DATE_A)
feat_c = add_features(raw_c, SNAPSHOT_DATE_C)
print(f"A: {len(feat_a):,} games, C: {len(feat_c):,} games")

#### Filter the rows (3.1.9-3.1.10)

##### 3.1.9 · Filter function

| # | Problem | Solution | Output |
|---|---|---|---|
| a | need to see how many games each filter removes | apply the filters one after another, count the games left after each (the order does not change the final table) | funnel table, shown in 3.1.12 |
| b | a missing value inside a filter | counts as failing the filter | missing date → removed |

**Purpose of `apply_filters`**
- A game is kept only if it passes **every** filter → the final table is the same in any order; one line (`df[f1 & f2 & ...]`) would give the same table.
- The function exists for the **funnel**: games left after each filter (shown in 3.1.12). It shows how 125,855 games become 47,603, and which filter removes the most → easy to explain and to spot a filter that removes too much.
- The number removed by a filter depends on its place in the list: it counts only games not removed by an earlier filter.
- Also used in 3.3.2 for the SQ2 panel (snapshot C).

In [ ]:
# 3.1.9 · Apply filters in order and count what each removes (problems a-b above)

def apply_filters(df, steps, table):
    """Apply filters in order and count the games left after each one.

    Args:
        df: table to filter.
        steps: list of (filter name, Series True = keep), applied in order.
        table: name written in the funnel, e.g. "games_clean".

    Returns:
        (kept, funnel): kept = copy of the rows that pass every filter;
        funnel = DataFrame with columns table, step, games_left (one row per step).
    """
    rows = [{"table": table, "step": "start", "games_left": len(df)}]
    keep = pd.Series(True, index=df.index)
    for name, mask in steps:
        keep &= mask.reindex(df.index).fillna(False)                               # 3.1.9b · missing = fails
        rows.append({"table": table, "step": name, "games_left": int(keep.sum())})   # 3.1.9a · count after each
    return df[keep].copy(), pd.DataFrame(rows)

##### 3.1.10 · The filters (both snapshots)

| # | Problem | Solution | Output |
|---|---|---|---|
| a | Chinese-first games | drop | `is_cn_game = True` → removed |
| b | Early Access games | drop | `is_early_access = True` → removed |
| c | below 10 reviews, reach is noise | keep >= 10 reviews | 3 reviews → removed |
| d | new games had no time to collect reviews | keep released >= 1 year before the snapshot | released 200 days before → removed |
| e | no genre, price or year → cannot form groups | keep games with all three | `genre_group = None` → removed |

- Not a higher review threshold because: it would filter on the outcome itself.

In [ ]:
# 3.1.10 · The filters shared by both snapshots (problems a-e above)

MIN_REVIEWS = 10              # 3.1.10c · below that, reach is noise
MIN_AGE_DAYS = 365            # 3.1.10d · every game had a year to collect reviews


def base_filters(df):
    """List the filters shared by both snapshots.

    Args:
        df: snapshot after add_features.

    Returns:
        List of (filter name, Series True = keep), in the order they are applied.
    """
    return [
        ("drop Chinese-first games (either of 2 signals)", ~df["is_cn_game"]),                  # 3.1.10a
        ("drop Early Access", ~df["is_early_access"]),                                         # 3.1.10b
        (f"keep >= {MIN_REVIEWS} reviews", df["total_reviews"] >= MIN_REVIEWS),                # 3.1.10c
        ("keep released >= 1 year before snapshot", df["days_since_release"] >= MIN_AGE_DAYS),   # 3.1.10d
        ("keep games with a genre group, price and year",                                       # 3.1.10e
         df["genre_group"].notna() & df["price"].notna() & df["release_year"].notna()),
    ]

#### Build the table (3.1.11-3.1.12)

##### 3.1.11 · Build `pcd_games_clean`

| # | Problem | Solution | Output |
|---|---|---|---|
| a | Traditional-Chinese-only games | drop (this table only) | `is_traditional_only = True` → removed |
| b | treatment is True/False | turn into 1/0 | `True` → `has_zh = 1` |
| c | review counts are very skewed | add the log10 of reviews | `1000` → `log_total_reviews = 3.0` |
| d | too many columns | keep the 12 the analysis uses, one row per game of A | 12 columns, sorted by `app_id` |

In [ ]:
# 3.1.11 · Build the cleaned dataset pcd_games_clean (problems a-d above)

def build_games_clean(feat_a):
    """Build pcd_games_clean from snapshot A.

    Args:
        feat_a: snapshot A after add_features.

    Returns:
        (games_clean, funnel): games_clean = one row per game that passed every filter,
        12 columns (app_id, name, genre_group, release_year, year_band, price, price_tier,
        n_other_langs, n_other_langs_bin, has_zh 0/1, total_reviews, log_total_reviews);
        funnel = games left after each filter.
    """
    steps = base_filters(feat_a) + [
        ("drop Traditional-Chinese-only", ~feat_a["is_traditional_only"]),         # 3.1.11a
    ]
    df, funnel = apply_filters(feat_a, steps, "games_clean")
    df["has_zh"] = df["has_zh_simplified"].astype(int)                           # 3.1.11b · True/False -> 1/0
    df["log_total_reviews"] = np.log10(df["total_reviews"]).round(6)             # 3.1.11c · rounded: same bytes every run
    for col in ["price_tier", "year_band", "n_other_langs_bin"]:
        df[col] = df[col].astype(int)                                            # no missing left after filters
    cols = [                                                                     # 3.1.11d · the 12 columns
        "app_id", "name", "genre_group", "release_year", "year_band", "price", "price_tier",
        "n_other_langs", "n_other_langs_bin", "has_zh", "total_reviews", "log_total_reviews",
    ]
    return df[cols].sort_values("app_id").reset_index(drop=True), funnel

##### 3.1.12 · Run: build the cleaned table and show the funnel

In [ ]:
# 3.1.12 · Run: build the cleaned dataset from snapshot A (the current catalogue, used for SQ1)

games_clean, funnel_a = build_games_clean(feat_a)
print(f"games_clean: {len(games_clean):,} games, {games_clean['has_zh'].sum():,} with Simplified Chinese")
display(funnel_a)                     # games left after each filter (3.1.9)
games_clean.head()

### 3.2 SQ1 groups → `pcd_sq1_groups`

- **Prepares for SQ1:** how much larger is the reach of games with Simplified Chinese, and in which genres?
- **Rule:** compare games with and without Chinese only inside groups of similar games.
- **Reason:** bigger games are both translated more and reviewed more → a direct comparison would mix the two.

#### 3.2.1 · Define the SQ1 comparison group

**Group = genre x price tier x year band x other languages**

| In the group | Reason |
|---|---|
| Genre | audience size and amount of text differ |
| Price tier | stands in for budget and studio size |
| Year band | older games had more time to collect reviews |
| Other languages | bigger games ship in more languages |

In [ ]:
# 3.2.1 · Define the SQ1 comparison group
# Reason: compare games only inside groups of similar games

# code -> label, to write readable group names ("Action | <=2017 | lt5 | 5+")
PRICE_LABEL = dict(enumerate(PRICE_LABELS, start=1))
YEAR_BAND_LABEL = dict(enumerate(YEAR_BAND_LABELS, start=1))
OTHER_LANGS_LABEL = dict(enumerate(OTHER_LANG_LABELS, start=1))


def add_sq1_group(games):
    """Copy of games with group_id = genre | year band | price tier | other languages."""
    # assign returns a copy with the new column; the input table is not changed
    return games.assign(group_id=(
        games["genre_group"] + " | " + games["year_band"].map(YEAR_BAND_LABEL) + " | "
        + games["price_tier"].map(PRICE_LABEL) + " | " + games["n_other_langs_bin"].map(OTHER_LANGS_LABEL)
    ))

#### 3.2.2 · Count games and median reviews per side

**Table `pcd_sq1_groups`** (one row per group)
- Per side: number of games and median reviews. Medians because: review counts are very skewed.
- `eligible` = >= 10 games per side because: fewer → one game moves the median.
- The comparison of the two sides is in 5.1.

In [ ]:
# 3.2.2 · Count games and median reviews on each side of each group
# Rules: medians because: review counts are very skewed
#        eligible = >= 10 games per side because: fewer -> one game moves the median

MIN_GAMES_PER_SIDE = 10       # a group is eligible only with this many games per side (also 3.3)


def build_sq1_groups(games):
    """One row per group: games and median reviews with / without Chinese, eligible."""
    keys = ["group_id", "genre_group", "year_band", "price_tier", "n_other_langs_bin"]
    wide = (
        games.groupby(keys + ["has_zh"])["total_reviews"]
        .agg(n="size", median="median")                    # count and median per group and side
        .unstack("has_zh")                                 # sides become columns: (stat, 0/1)
    )
    out = pd.DataFrame({
        "n_zh": wide[("n", 1)],
        "n_no_zh": wide[("n", 0)],
        "median_reviews_zh": wide[("median", 1)],
        "median_reviews_no_zh": wide[("median", 0)],
    }).reset_index()
    out[["n_zh", "n_no_zh"]] = out[["n_zh", "n_no_zh"]].fillna(0).astype(int)  # empty side -> 0
    out["eligible"] = (out["n_zh"] >= MIN_GAMES_PER_SIDE) & (out["n_no_zh"] >= MIN_GAMES_PER_SIDE)
    return out.sort_values(keys[1:]).reset_index(drop=True)   # genre, then codes: old -> new, cheap -> expensive

In [ ]:
# 3.2.3 · Run: build the SQ1 group table
# Reason: one row per group = the table SQ1 is analysed from (5.1)

sq1_groups = build_sq1_groups(add_sq1_group(games_clean))
print(f"sq1_groups: {len(sq1_groups)} groups, {sq1_groups['eligible'].sum()} eligible")
sq1_groups.head()

### 3.3 SQ2 groups → `pcd_sq2_groups`

- **Prepares for SQ2:** does adding Simplified Chinese after launch still increase reach, and in which genres?
- **Rule:** follow games with no Chinese in C (2024-01) to A (2026-06); compare those that added Chinese (treated) with similar games that did not (control).
- **Reason for a control:** every game gains reviews over time anyway.

#### 3.3.1 · Treated, control or excluded

- `treated` = added Simplified Chinese. `control` = added no Chinese and no major language.
- `excluded` = added another major language because: own localisation push → not a clean control.

In [ ]:
# 3.3.1 · Treated, control or excluded: what a game added between C and A
# Rules: treated = added Simplified Chinese; control = added no Chinese and no major language
# Excluded: added another major language because: own localisation push -> not a clean control

MAJOR_LANGS = [               # a control game must not have added any of these
    "German", "French", "Spanish - Spain", "Japanese", "Russian",
    "Korean", "Portuguese - Brazil",
]


# Languages in A but not in C = added after January 2024.
def label_arm(a, c):
    """treated / control / excluded for games without Chinese in C."""
    new_langs = [
        set(la.split("|")) - set(lc.split("|")) - {""}
        for la, lc in zip(a["all_langs"], c["all_langs"])
    ]
    added_zh = a["has_zh_simplified"].to_numpy()           # had no Chinese in C, has it in A
    added_major = np.array([bool(n & set(MAJOR_LANGS)) for n in new_langs])
    # np.select: first condition that is True gives the label, otherwise the default
    return np.select([added_zh, ~added_zh & ~added_major], ["treated", "control"], default="excluded")

#### 3.3.2 · Build the SQ2 panel

**Panel** (one row per game; not saved)
- Games: snapshot C, filtered as in 3.1, no Chinese in C, still in A.
- Characteristics from C because: the state before adding Chinese.
- 0 reviews in A → dropped because: delisted, -100% growth is not real.
- `growth = (reviews in A - reviews in C) / reviews in C`.

In [ ]:
# 3.3.2 · Build the SQ2 panel: the same games followed from C to A
# Rules: characteristics from C because: the state before adding Chinese
#        drop 0 reviews in A because: delisted -> -100% growth is not real

def build_sq2_panel(feat_a, feat_c):
    """One row per game of C (no Chinese, filtered as in 3.1) also present in A; and its funnel."""
    steps = base_filters(feat_c) + [
        ("keep games with no Chinese (Simplified or Traditional) in C",
         ~(feat_c["has_zh_simplified"] | feat_c["has_zh_traditional"])),
        ("keep games also present in A", feat_c["app_id"].isin(feat_a["app_id"])),
    ]
    c, funnel = apply_filters(feat_c, steps, "sq2_panel")
    c = c.set_index("app_id")
    a = feat_a.set_index("app_id").loc[c.index]            # same games, same order, from A

    p = pd.DataFrame(index=c.index)
    for col in ["name", "genre_group", "release_year", "year_band", "price_tier"]:
        p[col] = c[col]                                    # characteristics BEFORE: from C
    p["price_c"] = c["price"]
    p["reviews_c"] = c["total_reviews"]
    p["reviews_a"] = a["total_reviews"]
    p = p[p["reviews_a"] > 0].copy()                       # drop delisted games (0 reviews in A)
    a, c = a.loc[p.index], c.loc[p.index]
    p["growth"] = ((p["reviews_a"] - p["reviews_c"]) / p["reviews_c"]).round(6)

    p["arm"] = label_arm(a, c)                             # 3.3.1: treated / control / excluded
    for col in ["year_band", "price_tier"]:
        p[col] = p[col].astype(int)                        # no missing left after filters
    funnel = pd.concat([funnel, pd.DataFrame([
        {"table": "sq2_panel", "step": "keep games with reviews in A (not delisted)", "games_left": len(p)},
        {"table": "sq2_panel", "step": "arm = treated", "games_left": int((p["arm"] == "treated").sum())},
        {"table": "sq2_panel", "step": "arm = control", "games_left": int((p["arm"] == "control").sum())},
    ])], ignore_index=True)
    cols = ["name", "genre_group", "release_year", "year_band", "price_c", "price_tier",
            "reviews_c", "reviews_a", "growth", "arm"]
    return p[cols].reset_index().sort_values("app_id").reset_index(drop=True), funnel

In [ ]:
# 3.3.3 · Run: build the panel and show its funnel
# Reason: the funnel shows how many games each step keeps

sq2_panel, funnel_c = build_sq2_panel(feat_a, feat_c)
print(f"sq2_panel: {len(sq2_panel):,} games")
print(sq2_panel["arm"].value_counts().to_string())
display(funnel_c)

#### 3.3.4 · Count games and median growth per arm

**Group = genre x year band**

**Table `pcd_sq2_groups`**
- Per arm: number of games and median growth.
- `eligible` = >= 10 games per side.
- The comparison of the two arms is in 5.2.

In [ ]:
# 3.3.4 · Count games and median growth of each arm inside each SQ2 group
# Rule: group = genre x year band because: the strongest confounders of growth

def add_sq2_group(panel):
    """Copy of the panel with group_id_sq2 = genre | year band."""
    # assign returns a copy with the new column; map turns the year code back into its label
    return panel.assign(group_id_sq2=panel["genre_group"] + " | " + panel["year_band"].map(YEAR_BAND_LABEL))


def build_sq2_groups(panel):
    """One row per group: treated and control games, their median growth, eligible."""
    keys = ["group_id_sq2", "genre_group", "year_band"]
    used = panel[panel["arm"].isin(["treated", "control"])]          # "excluded" is not compared
    # count and median growth per group and arm; unstack makes treated / control columns
    wide = used.groupby(keys + ["arm"])["growth"].agg(n="size", median="median").unstack("arm")
    out = pd.DataFrame({
        "n_treated": wide[("n", "treated")],
        "n_control": wide[("n", "control")],
        "median_growth_treated": wide[("median", "treated")].round(6),
        "median_growth_control": wide[("median", "control")].round(6),
    }).reset_index()
    out[["n_treated", "n_control"]] = out[["n_treated", "n_control"]].fillna(0).astype(int)  # empty arm -> 0
    out["eligible"] = (out["n_treated"] >= MIN_GAMES_PER_SIDE) & (out["n_control"] >= MIN_GAMES_PER_SIDE)
    return out.sort_values(keys[1:]).reset_index(drop=True)          # genre, then old -> new

In [ ]:
# 3.3.5 · Run: build the SQ2 group table
# Reason: one row per group = the table SQ2 is analysed from (5.2)

sq2_groups = build_sq2_groups(add_sq2_group(sq2_panel))
print(f"sq2_groups: {len(sq2_groups)} groups, {sq2_groups['eligible'].sum()} eligible")
sq2_groups

### 3.4 Save the tables

- **Save:** write the three output tables to `data/processed/csv/`. (`data/processed/xlsx/` holds a copy to open and look at.)

In [ ]:
# 3.4.1 · Write the three tables as csv
# Rule: csv only because: every later step reads the csv
#   (data/processed/xlsx holds a copy of the same tables, only to open and look at)

tables = {
    "games_clean": games_clean,   # 3.1: cleaned data, one row per game
    "sq1_groups": sq1_groups,     # 3.2: one row per SQ1 group
    "sq2_groups": sq2_groups,     # 3.3: one row per SQ2 group
}
for name, table in tables.items():
    table.to_csv(PROCESSED_CSV + PREFIX + name + ".csv", index=False)
    print(f"{PREFIX}{name}: {len(table):,} rows x {table.shape[1]} columns")

## 4.​ Data analysis / EDA

- Describes the three saved tables only; no comparison of the two sides (that is section 5).
- Reads only the saved tables; figures go to `report/figures/`.
- To run it alone: cell I.a, then 4a.

| Section | Reads |
| :--- | :--- |
| 4.1 Games | `pcd_games_clean` |
| 4.2 SQ1 groups | `pcd_sq1_groups` |
| 4.3 SQ2 groups | `pcd_sq2_groups` |

In [ ]:
# 4a · Load the saved tables
# Reason: Stage 4 reads only the saved tables -> can run on its own

import matplotlib.pyplot as plt

games_clean = pd.read_csv(PROCESSED_CSV + PREFIX + "games_clean.csv")
sq1_groups = pd.read_csv(PROCESSED_CSV + PREFIX + "sq1_groups.csv")
sq2_groups = pd.read_csv(PROCESSED_CSV + PREFIX + "sq2_groups.csv")
print(f"games_clean {len(games_clean):,} rows | sq1_groups {len(sq1_groups)} | sq2_groups {len(sq2_groups)}")

### 4.1 Games: `pcd_games_clean`

### 4.2 SQ1 groups: `pcd_sq1_groups`

### 4.3 SQ2 groups: `pcd_sq2_groups`

## 5.​ Data modelling

- Compares the two sides of each group and answers SQ1, SQ2, SQ3.
- Reads the saved tables loaded in 4a; figures go to `report/figures/`.
- Exception: 5.1b also needs section 3 in memory (Run All).

| Section | Reads |
| :--- | :--- |
| 5.1 SQ1 | `pcd_sq1_groups` |
| 5.2 SQ2 | `pcd_sq2_groups` |
| 5.3 SQ3 | 5.1 and 5.2, per genre |

### 5.1 SQ1: how much larger is the reach of games with Simplified Chinese, and in which genres?

#### Does the SQ1 gap depend on the Chinese-first filter? (5.1b)

- **Reason:** the filter misses some Chinese games (Black Myth: Wukong).
- **Rule:** build `pcd_games_clean` and the SQ1 groups again with no filter, a stricter filter, and without the top 1% games; compute the gap each time.
- Gap of a group = median with Chinese / median without - 1; overall gap = median over eligible groups.
- Runs on copies → nothing is changed or saved.

In [ ]:
# 5.1b · Compute the SQ1 gap again under other filters
# Reason: the filter misses some Chinese games -> test whether that changes the result
# Rule: no filter, a stricter filter, and without the top 1% games
# Needs: feat_a and the functions of section 3 (Run All)

TOP_SHARE = 0.01              # share of most-reviewed games dropped in the last row


def sq1_gap(groups):
    """Median over eligible groups of (median with Chinese / median without - 1)."""
    eligible = groups[groups["eligible"]]
    pct_diff = eligible["median_reviews_zh"] / eligible["median_reviews_no_zh"] - 1
    return round(float(pct_diff.median()), 3)


# Works on a copy -> feat_a and the saved tables are not changed.
def sq1_summary(label, is_cn_game, drop_top=0.0):
    """SQ1 under the given Chinese-first flag, optionally without the most-reviewed games."""
    games, _ = build_games_clean(feat_a.assign(is_cn_game=is_cn_game))
    if drop_top:
        games = games[games["total_reviews"] <= games["total_reviews"].quantile(1 - drop_top)]
    groups = build_sq1_groups(add_sq1_group(games))
    return {"filter": label, "games": len(games), "eligible_groups": int(groups["eligible"].sum()),
            "median_gap": sq1_gap(groups)}


chosen = feat_a["cn_text"] | feat_a["cn_dev_name"]
stricter = chosen | feat_a["cn_dev_kana"] | feat_a["cn_audio"]
sensitivity = pd.DataFrame([
    sq1_summary("no filter", chosen & False),
    sq1_summary("chosen: text or developer name", chosen),
    sq1_summary("stricter: + kana names and voice-over", stricter),
    sq1_summary("chosen, without the 1% most-reviewed games", chosen, drop_top=TOP_SHARE),
])
sensitivity

### 5.2 SQ2: does adding Simplified Chinese after launch still increase reach, and in which genres?

### 5.3 SQ3: for each genre, translate at launch, later, or not a priority?

## 6.​ Model deployment